In [2]:
# Jax (Google) -> fastest, can use TPUs; implements NumPy-API (jax.numpy)

# Tensors
from jax import numpy as jnp

print(jnp.ones(shape=(2, 1)))
print(jnp.zeros(shape=(2, 1)))
print(jnp.array([1, 2, 3], dtype="float32"))

[[1.]
 [1.]]
[[0.]
 [0.]]
[1. 2. 3.]


In [3]:
# Random tensors (stateless -> we need to compute via function that is called - here with numpy)
import numpy as np


def apply_noise(x, seed):
    np.random.seed(seed)
    x = x * np.random.normal((3,))
    return x


x = 5
seed = 1337
y = apply_noise(x, seed)
seed += 1
z = apply_noise(x, seed)

In [4]:
# Random tensors (stateless -> we need to compute via function that is called - here via PRNGKey function from jax)

import jax

# Same seed key -> same result
seed_key = jax.random.PRNGKey(123)
print(jax.random.normal(seed_key, shape=(3,)))
print(jax.random.normal(seed_key, shape=(3,)))

# Split to create new seed_key from existing one -> deterministic split (always the same, also true for sequences)
new_seed_key = jax.random.split(seed_key, num=1)[0]
print(jax.random.normal(new_seed_key, shape=(3,)))
new_seed_key = jax.random.split(seed_key, num=1)[0]
print(jax.random.normal(new_seed_key, shape=(3,)))


[1.6359469  0.8408094  0.02212393]
[1.6359469  0.8408094  0.02212393]
[-0.49093357 -0.9478693  -1.775197  ]
[-0.49093357 -0.9478693  -1.775197  ]


In [5]:
# Tensor assignment (tensors not mutable like in tensorflow -> create new from old and change value)

x = jnp.array([1, 2, 3], dtype="float32")
print(x)
x_new = x.at[0].set(10)  # Copy and change one value
print(x_new)

[1. 2. 3.]
[10.  2.  3.]


In [6]:
# Tensor operations

a = jnp.ones((2, 2))
b = jnp.ones((2, 2))
c = a + b
print(c)
d = jnp.square(c)
print(d)
e = jnp.sqrt(d)
print(e)
f = b + c
g = jnp.matmul(a, b)
g *= f

[[2. 2.]
 [2. 2.]]
[[4. 4.]
 [4. 4.]]
[[2. 2.]
 [2. 2.]]


In [14]:
# Linear Classifier

learning_rate = 0.1
num_samples_per_class = 1000

negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class)

# Put all in one array
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

targets = np.vstack((np.zeros((num_samples_per_class, 1), dtype="float32"),
                     np.ones((num_samples_per_class, 1), dtype="float32")))

def model(inputs, W, b):
    return jnp.matmul(inputs, W) + b

def mean_squared_error(targets, predictions):
    per_sample_losses = jnp.square(targets - predictions)
    return jnp.mean(per_sample_losses)

def compute_loss(state, inputs, targets):
    W, b = state
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    return loss

grad_fn = jax.value_and_grad(compute_loss)

@jax.jit
def training_step(inputs, targets, W, b):
    loss, grads = grad_fn((W, b), inputs, targets)
    grad_wrt_W, grad_wrt_b = grads
    W = W - grad_wrt_W * learning_rate
    b = b - grad_wrt_b * learning_rate
    return loss, W, b

input_dim = 2
output_dim = 1
W = jax.numpy.array(np.random.uniform(size=(input_dim, output_dim)))
b = jax.numpy.array(np.zeros(shape=(output_dim,)))
state = (W, b)
for step in range(400):
    loss, W, b = training_step(inputs, targets, W, b)
    print(f"Loss at step {step}: {loss:.4f}")
    print(f"W: {W}, b: {b}")

Loss at step 0: 5.1395
W: [[ 0.11793019]
 [-0.19351476]], b: [-0.36319906]
Loss at step 1: 0.9915
W: [[0.42470664]
 [0.1227868 ]], b: [-0.1669034]
Loss at step 2: 0.2939
W: [[ 0.28843865]
 [-0.0183846 ]], b: [-0.20108376]
Loss at step 3: 0.1685
W: [[0.33287343]
 [0.0280557 ]], b: [-0.14328349]
Loss at step 4: 0.1391
W: [[ 0.30407894]
 [-0.00152176]], b: [-0.1249329]
Loss at step 5: 0.1264
W: [[0.30562288]
 [0.0003761 ]], b: [-0.09233879]
Loss at step 6: 0.1172
W: [[ 0.29524177]
 [-0.01011629]], b: [-0.0673202]
Loss at step 7: 0.1091
W: [[ 0.29015437]
 [-0.01512675]], b: [-0.04089975]
Loss at step 8: 0.1018
W: [[ 0.28331846]
 [-0.02196387]], b: [-0.01666479]
Loss at step 9: 0.0952
W: [[ 0.27758834]
 [-0.02766464]], b: [0.006919]
Loss at step 10: 0.0891
W: [[ 0.2717825 ]
 [-0.03345486]], b: [0.02929376]
Loss at step 11: 0.0836
W: [[ 0.26636633]
 [-0.03885103]], b: [0.050752]
Loss at step 12: 0.0785
W: [[ 0.26113415]
 [-0.04406612]], b: [0.07123653]
Loss at step 13: 0.0738
W: [[ 0.2561548